# P02
## 상담봇
### State
```py
ChatState

user_input: str  # 사용자 메세지
sentiment: str (positive, negative, aggressive)
core_msg: str
warning_count: int  # 상담할때, 공격적인 언행을 하면 경고 누적 횟수
response: str  # 답변
```
### Node
- `block_user`: 제공
- `analyze_sentiment`: LLM이 `user_input` 을 분석하여 `sentiment`를 `['positive', 'negative', 'agreesive']` 중 한가지로만 세팅
- `positive_node`: 제공
- `negative_node`: 제공
- `aggressive_node`: 제공
- `make_final_msg`: LLM이 `core_msg` 를 보고 최종 답변 생성

### Router
- `block_router` : `state['warning_count']`를 확인(3이상인지)하고 다음 노드 결정
- `emotion_router`: 단순히 `state['sentiment']`를 보고 다음 노드 결정

### Flow
```mermaid
flowchart LR
    START([START]) --> block_router{block_router}
    
    %% START 분기
    block_router -->|gogo| analyze_sentiment[analyze_sentiment]
    block_router -->|block| block_user[block_user]
    
    %% block_user 종료
    block_user --> END([END])
    
    %% 감정 분석 후 분기
    analyze_sentiment --> emotion_router{emotion_router}
    emotion_router -->|positive| positive_node[positive_node]
    emotion_router -->|negative| negative_node[negative_node]
    emotion_router -->|aggressive| aggressive_node[aggressive_node]
    
    %% 최종 메시지 생성으로 모임
    positive_node --> make_final_msg[make_final_msg]
    negative_node --> make_final_msg[make_final_msg]
    aggressive_node --> make_final_msg[make_final_msg]
    
    %% 최종 메시지 종료
    make_final_msg --> END
```

In [ ]:
from typing import TypedDict, Literal

class ChatState(TypedDict):
    user_input: str
    sentiment: Literal['positive', 'negative', 'aggressive']  # 말그래도, 다음에 오는것들 중 하나! 객관식
    core_msg: str
    warning_count: int  
    response: str

In [ ]:
import random
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4.1-mini')


def analyze_sentiment(state: ChatState):
    prompt = f'''
    아래 문장의 감정을 분류. 3가지 중 하나로만 판단해라
    - positive
    - negative
    - aggressive

    문장: {state['user_input']}
    '''
    result = llm.invoke(prompt).content  # AIMessage 기때문에, .content
    return {'sentiment': result}


def positive_node(state: ChatState):
    msgs = ['최고', '멋져', '훌륭']
    keyword = random.choice(msgs)
    return {'core_msg': keyword}


def negative_node(state: ChatState):
    msgs = ['힘내', '위로', '괜찮']
    keyword = random.choice(msgs)
    return {'core_msg': keyword}


def aggressive_node(state: ChatState):
    # dict.get(a, b)  -> key a가 있으면, 해당 value. key a가 없으면 b가 나옴
    count = state.get('warning_count', 0) + 1  # state(dict)에 'warning_count' 키가 있으면, 그대로 사용. 없으면 0
    return {'core_msg':  '공격적인 표현은 삼가라', 'warning_count': count}


def block_user(state: ChatState):
    return {'response': '님 차단'}


def make_final_msg(state: ChatState):
    prompt = f'''
    넌 상담 선생님이야.
    다음 사용자 입력에 맞는 답변을
    핵심 키워드를 참조해서 만들어줘,

    사용자 입력: {state['user_input']}
    핵심 키워드: {state['core_msg']}
    '''

    result = llm.invoke(prompt).content
    return {'response': result}

In [ ]:
def block_router(state: ChatState):
    if state.get('warning_count', 0) >= 3:
        return 'block'
    else:
        return 'gogo'


def emotion_router(state: ChatState):
    return state['sentiment']

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)
# Node 등록할때 이름 안적으면, 함수명이 곧 이름이 된다.
builder.add_node(analyze_sentiment)
builder.add_node(block_user)
builder.add_node(positive_node)
builder.add_node(negative_node)
builder.add_node(aggressive_node)
builder.add_node(make_final_msg)

# 엣지 조립
builder.add_conditional_edges(
    START,
    block_router,
    {
        'gogo': 'analyze_sentiment',
        'block': 'block_user'
    }
)
builder.add_edge('block_user', END)
builder.add_conditional_edges(
    'analyze_sentiment',
    emotion_router,
    {
        'positive': 'positive_node',
        'negative': 'negative_node',
        'aggressive': 'aggressive_node'
    }
)
builder.add_edge('positive_node', 'make_final_msg')
builder.add_edge('negative_node', 'make_final_msg')
builder.add_edge('aggressive_node', 'make_final_msg')
builder.add_edge('make_final_msg', END)

graph = builder.compile()
graph

In [ ]:
result = graph.invoke({
    'user_input': '나 오늘 야근이야. 죽고싶어. 모든 세상을 부셔버리고 불살라 버리겠다',
    'warning_count': 3,
})

result